# 04 · Feature engineering
All features are computed at issue time *t* from data ≤ *t*; target-time calendar features are known in advance. The leakage test in `backend/tests/test_ml_pipeline.py` enforces this.

In [1]:
import sys, pathlib
ROOT = pathlib.Path.cwd().parent
sys.path.insert(0, str(ROOT / "backend"))
import pandas as pd, numpy as np, json
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30)

In [2]:
from airsense.ml.feature_engineering import build_features, add_targets, base_feature_columns
h = pd.read_parquet(ROOT/'data/processed/hourly_measurements.parquet')
w = pd.read_parquet(ROOT/'data/interim/weather_hourly.parquet')
st = h[h.location_id==3409526]
f = build_features(st, w[w.location_id==3409526])
f = add_targets(f, st.set_index('ts')['pm25'], st.set_index('ts')['pm25_imputed'], horizons=[1,6,24])
print(len(base_feature_columns(f)), 'features'); f.tail(3).T.head(40)

57 features


ts,2026-11-19 16:00:00+00:00,2026-11-19 17:00:00+00:00,2026-11-19 18:00:00+00:00
pm25_lag0,66.150000,72.725000,NaN
pm25_lag1,62.750000,66.150000,72.725000
pm25_lag2,56.075000,62.750000,66.150000
pm25_lag3,53.475000,56.075000,62.750000
pm25_lag6,58.050000,53.550000,54.825000
pm25_lag12,93.550000,81.750000,73.700000
pm25_lag24,118.866667,100.825000,96.645833
pm25_lag48,78.500000,82.550000,83.641667
pm25_lag168,62.050000,75.975000,77.904167
pm25_rmean3,61.658333,67.208333,69.437500


In [3]:
# Leakage check: perturb the future, features at t must not change
t = f.index[5000]
st2 = st.copy(); st2.loc[st2.ts > t, 'pm25'] = 999
f2 = build_features(st2, w[w.location_id==3409526])
print('identical features at t:', f.loc[t, base_feature_columns(f)].equals(f2.loc[t, base_feature_columns(f)]))

identical features at t: True
